# Notebook 02 — Entrenamiento Contrastivo (SupCon)

**Objetivo:** Entrenar el encoder ResNet18 con Supervised Contrastive Learning para producir embeddings de 1024 dims.

**IMPORTANTE:** Este notebook está diseñado para ejecutarse en **Google Colab con GPU**.
En CPU local toma ~2h por epoch con batch_size=32. En Colab T4 toma ~2-5 min por epoch con batch_size=256.

## Cómo ejecutar en Colab
1. Runtime → Change runtime type → **T4 GPU** (gratis) o A100 (Pro)
2. Descomenta las celdas marcadas `# COLAB`
3. Runtime → Run all
4. Al terminar: descarga `artifacts/checkpoints/encoder_best.pt` para continuar localmente

In [ ]:
# ════════════════════════════════════════════════════════════════
# SETUP — Local o Colab (auto-detección)
# Pre-requisito Colab: Secrets GITHUB_TOKEN, GITHUB_USER, GITHUB_EMAIL
# Pre-requisito Colab: Runtime → Change runtime type → T4 GPU
# ════════════════════════════════════════════════════════════════
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata, drive

    GH_TOKEN = userdata.get("GITHUB_TOKEN")
    GH_USER  = userdata.get("GITHUB_USER")
    GH_EMAIL = userdata.get("GITHUB_EMAIL")
    REPO     = "Malaria-Dectetion-Deeplearning"
    REPO_URL = f"https://{GH_TOKEN}@github.com/{GH_USER}/{REPO}.git"

    if not os.path.exists(f"/content/{REPO}"):
        get_ipython().system(f"git clone -q {REPO_URL}")
    get_ipython().run_line_magic("cd", f"/content/{REPO}")
    get_ipython().system(f'git config user.email "{GH_EMAIL}"')
    get_ipython().system(f'git config user.name  "{GH_USER}"')
    get_ipython().system("git pull -q origin main")
    get_ipython().system("pip install -r requirements.txt -q")

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/malaria_project"
    get_ipython().system(f"mkdir -p {DRIVE_ROOT}/checkpoints {DRIVE_ROOT}/embeddings")
    get_ipython().system("rm -rf artifacts/checkpoints data/embeddings")
    get_ipython().system("mkdir -p artifacts data")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/checkpoints artifacts/checkpoints")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/embeddings   data/embeddings")
    get_ipython().system("mkdir -p artifacts/figures artifacts/metrics artifacts/logs data/processed")
    print("✓ Colab listo. Pesados → Drive, ligeros → repo.")

cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / "src").exists() else cwd.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"Working dir: {REPO_ROOT}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# DATASET — Kaggle API (solo si falta cell_images/)
# ════════════════════════════════════════════════════════════════
if IN_COLAB and not os.path.exists("cell_images"):
    if os.path.exists(f"{DRIVE_ROOT}/kaggle.json"):
        get_ipython().system("mkdir -p ~/.kaggle")
        get_ipython().system(f"cp {DRIVE_ROOT}/kaggle.json ~/.kaggle/")
    else:
        from google.colab import files
        print("Sube tu kaggle.json (Kaggle → Settings → Create API Token):")
        files.upload()
        get_ipython().system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/")
        get_ipython().system(f"cp ~/.kaggle/kaggle.json {DRIVE_ROOT}/kaggle.json")
    get_ipython().system("chmod 600 ~/.kaggle/kaggle.json")
    get_ipython().system("pip install kaggle -q")
    get_ipython().system("kaggle datasets download -d iarunava/cell-images-for-detecting-malaria -q")
    get_ipython().system("unzip -q cell-images-for-detecting-malaria.zip")
    get_ipython().system("rm -f cell-images-for-detecting-malaria.zip")
    get_ipython().system("if [ -d cell_images/cell_images ]; then mv cell_images/cell_images/* cell_images/ 2>/dev/null; rmdir cell_images/cell_images 2>/dev/null; fi")
    print(f"✓ Dataset listo: {len(os.listdir('cell_images/Parasitized'))} parasitized, "
          f"{len(os.listdir('cell_images/Uninfected'))} uninfected")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from src.utils.seed import set_global_seed
from src.utils.io import load_config, load_json
from src.data.split import make_stratified_split
from src.data.augmentations import get_contrastive_transform
from src.data.dataset import SupConPairDataset
from src.models.encoder import ContrastiveEncoder
from src.training.train_contrastive import train
from src.visualization.training_plots import plot_training_curves

import matplotlib.pyplot as plt
%matplotlib inline

set_global_seed(42)

In [ ]:
cfg = load_config('configs/contrastive.yaml')
data_cfg = load_config('configs/data.yaml')
print('Configuración del encoder:', cfg['encoder'])
print('Épocas:', cfg['training']['epochs'])

## 1. Preparar datos

In [ ]:
from torch.utils.data import DataLoader

# Generar splits si no existen
train_df, val_df, test_df = make_stratified_split(
    dataset_root=data_cfg['dataset_root'],
    processed_dir=data_cfg['processed_dir'],
    seed=data_cfg['seed'],
)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

use_gpu = torch.cuda.is_available()
aug_cfg = cfg.get('augmentations', {})
transform = get_contrastive_transform(
    img_size=aug_cfg.get('img_size', 96),
    blur_prob=aug_cfg.get('blur_prob', 0.3),
)

batch_size = cfg['training']['batch_size_gpu'] if use_gpu else cfg['training']['batch_size_cpu']
num_workers = 2 if use_gpu else 0
print(f'batch_size={batch_size} | num_workers={num_workers} | GPU={use_gpu}')

train_ds = SupConPairDataset('data/processed/train.csv', transform=transform)
val_ds   = SupConPairDataset('data/processed/val.csv',   transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=use_gpu, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=use_gpu)

## 2. Entrenamiento SupCon

In [ ]:
from pathlib import Path
Path('artifacts/checkpoints').mkdir(parents=True, exist_ok=True)

history = train(
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir='artifacts/checkpoints',
)
print('\nEntrenamiento completado!')
print(f'Mejor val_loss: {min(history["val"]):.4f} (epoch {history["val"].index(min(history["val"]))+1})')

In [ ]:
fig = plot_training_curves(history, save_path='artifacts/figures/training_curves.png')
plt.show()

import json
Path('artifacts/logs').mkdir(parents=True, exist_ok=True)
with open('artifacts/logs/contrastive_history.json', 'w') as f:
    json.dump(history, f)

In [ ]:
# En Colab, encoder_best.pt YA está en Drive vía el simlink
# (artifacts/checkpoints/ → /content/drive/MyDrive/malaria_project/checkpoints/)
# El NB03 lo leerá automáticamente desde la misma ruta.
if IN_COLAB:
    ckpt = Path("artifacts/checkpoints/encoder_best.pt")
    if ckpt.exists():
        size_mb = ckpt.stat().st_size / 1e6
        print(f"✓ Checkpoint en Drive: {ckpt.resolve()} ({size_mb:.1f} MB)")
    else:
        print("⚠ No se encontró encoder_best.pt — ¿terminó el entrenamiento?")

In [ ]:
# COLAB — descomenta para descargar el checkpoint a tu PC
# from google.colab import files
# files.download('artifacts/checkpoints/encoder_best.pt')

# O guardar en Drive:
# import shutil
# shutil.copy('artifacts/checkpoints/encoder_best.pt',
#             '/content/drive/MyDrive/malaria_encoder_best.pt')

In [ ]:
# ════════════════════════════════════════════════════════════════
# SYNC CON GITHUB — push del notebook ejecutado + figuras + logs
# encoder_best.pt vive en Drive (excluido por .gitignore)
# ════════════════════════════════════════════════════════════════
if IN_COLAB:
    NOTEBOOK = "02_contrastive_training"
    try:
        from google.colab import _message
        _message.blocking_request("save_notebook", request="", timeout_sec=10)
    except Exception:
        pass
    get_ipython().system(
        "git add notebooks/{nb}.ipynb artifacts/figures artifacts/metrics artifacts/logs data/processed".format(nb=NOTEBOOK)
    )
    get_ipython().system(f'git commit -m "nb {NOTEBOOK}: entrenamiento contrastivo ejecutado" || echo "Sin cambios para commitear"')
    get_ipython().system("git push -q origin main && echo '✓ Pushed a GitHub' || echo '⚠ Push falló (revisa GITHUB_TOKEN)'")

## Siguiente paso
Coloca `encoder_best.pt` en `artifacts/checkpoints/` y ejecuta:
```bash
python -m scripts.extract_embeddings --checkpoint artifacts/checkpoints/encoder_best.pt
```
O abre `notebooks/03_extract_embeddings.ipynb`.